# Fair, Interpretable, and Actionable Readmission Risk Modeling
## Diabetes 130-US Hospitals (1999–2008)

**StudyBuild — Project 02 | Digital Health & Medical Data Science Track**

This notebook provides a complete, reproducible pipeline answering all nine clinical questions (Q1–Q9) required by the project guide.

**Team approach summary**
- Target: Binary classification (`<30` vs. Not) for clinical actionability + multi-class exploration
- Baseline models: Logistic Regression (interpretable) + Decision Tree
- Optional stronger model: Gradient Boosting (justified)
- Strict leakage prevention via patient-level awareness
- Full fairness analysis across race, gender, age
- All figures saved to `../figures/`


## 0. Setup & Reproducibility

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    recall_score, f1_score, precision_score, accuracy_score,
    roc_auc_score, RocCurveDisplay
)
from sklearn.inspection import permutation_importance

# Paths
DATA_PATH = Path('../data/diabetic_data.csv')
FIGURES_PATH = Path('../figures')
FIGURES_PATH.mkdir(exist_ok=True, parents=True)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries loaded successfully.')
print(f'Random seed fixed at {RANDOM_STATE}')


## 1. Data Loading & Initial Inspection

**Corresponding to: Data Preparation (آماده‌سازی داده‌ها)**

We load the dataset treating `?` as missing values, as documented on the UCI page.


In [ ]:
df_raw = pd.read_csv(DATA_PATH, na_values='?', low_memory=False)
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.shape[1]}')
print(f'\nFirst 5 rows:')
df_raw.head()


In [ ]:
print('Data types and non-null counts:')
df_raw.info()


## Q1 — What does the readmission population look like?

**Phase: Data Preparation + EDA**

We describe the sample, the distribution of the target, demographics, key utilization variables, and flag unusual / highly missing columns.


In [ ]:
# Target distribution
print('=== Target: readmitted ===')
print(df_raw['readmitted'].value_counts())
print('\nProportions:')
print(df_raw['readmitted'].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
order = ['<30', '>30', 'NO']
counts = df_raw['readmitted'].value_counts().reindex(order)
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette=['#e74c3c', '#f39c12', '#2ecc71'])
ax.set_title('Distribution of Readmission Classes')
ax.set_xlabel('Readmitted')
ax.set_ylabel('Number of Encounters')
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f'{v:,}\n({v/len(df_raw)*100:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nInterpretation: The early-readmission class (<30) is a clear minority (~11.2%). Class imbalance must be handled carefully in modeling and evaluation.')


In [ ]:
# Demographics
print('=== Race ===')
print(df_raw['race'].value_counts(dropna=False))
print('\n=== Gender ===')
print(df_raw['gender'].value_counts(dropna=False))
print('\n=== Age brackets ===')
print(df_raw['age'].value_counts().sort_index())


In [ ]:
# Key utilization variables
util_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
             'num_medications', 'number_outpatient', 'number_emergency',
             'number_inpatient', 'number_diagnoses']
print(df_raw[util_cols].describe().round(2))


In [ ]:
# Missingness overview
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
print('Top missing columns:')
print(missing_df[missing_df['Missing'] > 0].head(12))

print('\n=== Unusual / high-missing columns ===')
print('- weight: ~96.9% missing → drop for modeling')
print('- max_glu_serum & A1Cresult: very high missingness (lab not always ordered)')
print('- medical_specialty & payer_code: high missingness; keep with careful encoding or drop')
print('- race: ~2.2% missing → impute as "Unknown" or keep as category')


In [ ]:
# Repeat encounters check (critical for leakage)
print(f'Total encounters: {len(df_raw):,}')
print(f'Unique patients (patient_nbr): {df_raw["patient_nbr"].nunique():,}')
print(f'Encounters from patients with >1 visit: {(df_raw.duplicated(subset=["patient_nbr"], keep=False)).sum():,}')
print('\nNote: Same patient can appear multiple times. We will handle this carefully in the train/test split to reduce leakage.')


## Q2 — Which measurements differ most across readmission groups?

**Phase: Data Preparation / EDA**


In [ ]:
# Create a clean copy for EDA
df = df_raw.copy()

# Binary target for many analyses
df['early_readmit'] = (df['readmitted'] == '<30').astype(int)

# Summary by readmission group
group_cols = ['time_in_hospital', 'num_medications', 'num_lab_procedures',
              'number_inpatient', 'number_outpatient', 'number_emergency', 'number_diagnoses']

print('=== Mean values by readmitted class ===')
print(df.groupby('readmitted')[group_cols].mean().round(2).reindex(['<30', '>30', 'NO']))


In [ ]:
# Visualizations for Q2
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='readmitted', y='time_in_hospital', order=['<30','>30','NO'], ax=axes[0,0])
axes[0,0].set_title('Time in Hospital by Readmission')

sns.boxplot(data=df, x='readmitted', y='num_medications', order=['<30','>30','NO'], ax=axes[0,1])
axes[0,1].set_title('Number of Medications by Readmission')

sns.boxplot(data=df, x='readmitted', y='number_inpatient', order=['<30','>30','NO'], ax=axes[1,0])
axes[1,0].set_title('Prior Inpatient Visits by Readmission')

sns.boxplot(data=df, x='readmitted', y='number_diagnoses', order=['<30','>30','NO'], ax=axes[1,1])
axes[1,1].set_title('Number of Diagnoses by Readmission')

plt.tight_layout()
plt.savefig(FIGURES_PATH / '02_utilization_by_readmission.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Patients with early readmission (<30) tend to have slightly longer stays, more medications, more prior inpatient visits, and more diagnoses. Prior utilization appears particularly informative.')


## Q3 — Which variables appear most strongly associated with 30-day readmission?

**Phase: EDA**

We examine associations while emphasizing that observed associations do **not** imply causation.


In [ ]:
# Simple association checks
print('=== Early readmission rate by selected variables ===')

# Age
print('\nBy Age:')
print(df.groupby('age')['early_readmit'].mean().sort_values(ascending=False).round(3))

# Gender
print('\nBy Gender:')
print(df.groupby('gender')['early_readmit'].mean().round(3))

# Race
print('\nBy Race:')
print(df.groupby('race')['early_readmit'].mean().sort_values(ascending=False).round(3))

# DiabetesMed
print('\nBy diabetesMed:')
print(df.groupby('diabetesMed')['early_readmit'].mean().round(3))

# Insulin
print('\nBy insulin:')
print(df.groupby('insulin')['early_readmit'].mean().round(3))


In [ ]:
# Correlation-style view for numeric features
numeric_for_corr = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_emergency',
                    'number_inpatient', 'number_diagnoses', 'early_readmit']
corr = df[numeric_for_corr].corr()['early_readmit'].drop('early_readmit').sort_values(key=abs, ascending=False)
print('Pearson correlation with early_readmit:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Association Strength (Pearson) with Early Readmission')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '03_numeric_associations.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: number_inpatient shows the strongest linear association among numeric features. This is expected (prior utilization predicts future utilization) but is not causal proof.')


## Data Preparation for Modeling

**Phase: Data Preparation (آماده‌سازی داده‌ها)**

Key decisions:
1. Drop high-missing columns (`weight`, `max_glu_serum`, `A1Cresult`, `medical_specialty`, `payer_code`)
2. Drop identifiers (`encounter_id`) after leakage checks
3. Keep `patient_nbr` only for stratified/patient-aware considerations, then exclude from features
4. Group ICD-9 codes into broader categories
5. Create binary target for primary modeling
6. Handle remaining missing values
7. Encode categoricals


In [ ]:
def map_diag(code):
    """Map ICD-9 codes to broad clinical categories (simplified)."""
    if pd.isna(code):
        return 'Missing'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        code_num = float(code)
    except:
        return 'Other'
    if 390 <= code_num <= 459 or code_num == 785:
        return 'Circulatory'
    elif 460 <= code_num <= 519 or code_num == 786:
        return 'Respiratory'
    elif 520 <= code_num <= 579 or code_num == 787:
        return 'Digestive'
    elif 250 <= code_num < 251:
        return 'Diabetes'
    elif 800 <= code_num <= 999:
        return 'Injury'
    elif 710 <= code_num <= 739:
        return 'Musculoskeletal'
    elif 580 <= code_num <= 629 or code_num == 788:
        return 'Genitourinary'
    elif 140 <= code_num <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

df['diag1_cat'] = df['diag_1'].apply(map_diag)
df['diag2_cat'] = df['diag_2'].apply(map_diag)
df['diag3_cat'] = df['diag_3'].apply(map_diag)

print(df['diag1_cat'].value_counts())


In [ ]:
# Feature selection
drop_cols = [
    'encounter_id', 'patient_nbr', 'weight', 'max_glu_serum', 'A1Cresult',
    'medical_specialty', 'payer_code', 'diag_1', 'diag_2', 'diag_3',
    'readmitted',  # we create our own targets
    'examide', 'citoglipton'  # almost constant
]

# Medication columns that are almost always "No"
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']

# Keep only the most informative ones + change + diabetesMed
keep_meds = ['metformin', 'insulin', 'change', 'diabetesMed']

feature_cols = [
    'race', 'gender', 'age',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses',
    'diag1_cat', 'diag2_cat', 'diag3_cat'
] + keep_meds

# Clean race
df['race'] = df['race'].fillna('Unknown')

# Binary target
df['target_binary'] = (df['readmitted'] == '<30').astype(int)

# Multi-class target (optional)
df['target_multi'] = df['readmitted'].map({'<30': 0, '>30': 1, 'NO': 2})

print('Selected features:', len(feature_cols))
print(feature_cols)


In [ ]:
# Final modeling dataframe
df_model = df[feature_cols + ['target_binary', 'target_multi', 'patient_nbr']].copy()

# Simple imputation for any remaining NaNs in categoricals
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = df_model[col].fillna('Unknown')

print('Final shape:', df_model.shape)
print('Missing values left:', df_model.isnull().sum().sum())
print('\nBinary target distribution:')
print(df_model['target_binary'].value_counts(normalize=True).round(4))


### Train / Test Split (Leakage-aware)

Because the same patient can have multiple encounters, a pure random split can leak information.  
We perform a stratified split on the encounter level while documenting the limitation.  
For a stricter approach one could split by unique `patient_nbr`; here we use stratified encounter-level split with fixed seed for reproducibility (as allowed by the project).


In [ ]:
X = df_model[feature_cols]
y = df_model['target_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train):,}  |  Positive rate: {y_train.mean():.3f}')
print(f'Test size:  {len(X_test):,}  |  Positive rate: {y_test.mean():.3f}')


In [ ]:
# Identify column types
numeric_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_emergency',
                    'number_inpatient', 'number_diagnoses',
                    'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

categorical_features = [c for c in feature_cols if c not in numeric_features]

print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)


## Q4 — Can a baseline model predict readmission risk?

**Phase: Modeling & Evaluation (مدل‌سازی و ارزیابی)**

We train two strong baselines:
1. **Logistic Regression** (highly interpretable coefficients)
2. **Decision Tree** (interpretable rules + feature importance)

We also train a **Gradient Boosting** model as an optional stronger comparator (justified by better handling of non-linearities and interactions while still allowing permutation importance).


In [ ]:
# Logistic Regression pipeline
log_reg = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, C=0.5))
])

log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Not <30', '<30']))


In [ ]:
# Decision Tree
tree = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                   class_weight='balanced', random_state=RANDOM_STATE))
])

tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print('=== Decision Tree (max_depth=6) ===')
print(classification_report(y_test, y_pred_tree, target_names=['Not <30', '<30']))


In [ ]:
# Optional stronger model: Gradient Boosting
gb = Pipeline([
    ('prep', preprocessor),
    ('clf', GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.08,
                                       random_state=RANDOM_STATE, subsample=0.8))
])

gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

print('=== Gradient Boosting (optional stronger model) ===')
print(classification_report(y_test, y_pred_gb, target_names=['Not <30', '<30']))
print('\nJustification: Gradient Boosting captures non-linear interactions (e.g., prior inpatient × age × medications) that linear models miss, while still allowing post-hoc interpretation via permutation importance. It is appropriate for tabular clinical data of this size.')


## Q5 — Is overall accuracy enough?

**Phase: Modeling & Evaluation**

No. Because of severe class imbalance, accuracy is misleading. We focus on **Recall for the <30 class** (sensitivity to early readmissions) and F1.


In [ ]:
def evaluate_model(y_true, y_pred, y_prob, name):
    print(f'\n========== {name} ==========')
    print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'Recall (<30): {recall_score(y_true, y_pred):.4f}')
    print(f'Precision (<30): {precision_score(y_true, y_pred):.4f}')
    print(f'F1 (<30): {f1_score(y_true, y_pred):.4f}')
    if y_prob is not None:
        print(f'ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}')
    
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not <30', '<30'])
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix — {name}')
    plt.savefig(FIGURES_PATH / f'05_cm_{name.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()
    return cm

cm_lr = evaluate_model(y_test, y_pred_lr, y_prob_lr, 'Logistic Regression')
cm_tree = evaluate_model(y_test, y_pred_tree, None, 'Decision Tree')
cm_gb = evaluate_model(y_test, y_pred_gb, y_prob_gb, 'Gradient Boosting')


In [ ]:
print('''
Clinical note on false negatives:
Missing an early-readmission case (False Negative) means the hospital may fail to provide enhanced discharge planning, 
medication reconciliation, or early follow-up to a high-risk diabetic patient. 
This can lead to preventable complications, higher costs, and worse outcomes.
Therefore Recall for the <30 class is prioritized over overall accuracy.
''')


## Q6 — Where does the model fail?

**Phase: Modeling & Evaluation**


In [ ]:
# Error analysis on test set using Logistic Regression (most interpretable)
X_test_err = X_test.copy()
X_test_err['y_true'] = y_test.values
X_test_err['y_pred'] = y_pred_lr
X_test_err['error_type'] = 'Correct'
X_test_err.loc[(X_test_err['y_true']==1) & (X_test_err['y_pred']==0), 'error_type'] = 'False Negative'
X_test_err.loc[(X_test_err['y_true']==0) & (X_test_err['y_pred']==1), 'error_type'] = 'False Positive'

print('Error type counts:')
print(X_test_err['error_type'].value_counts())

print('\n=== False Negatives: mean of key variables ===')
print(X_test_err[X_test_err['error_type']=='False Negative'][['time_in_hospital','num_medications','number_inpatient','number_diagnoses']].mean().round(2))

print('\n=== Correct Positives (for comparison) ===')
print(X_test_err[(X_test_err['y_true']==1) & (X_test_err['y_pred']==1)][['time_in_hospital','num_medications','number_inpatient','number_diagnoses']].mean().round(2))


In [ ]:
# Error rates by age and race
print('False Negative rate by Age (among true <30):')
fn_by_age = X_test_err[X_test_err['y_true']==1].groupby('age')['error_type'].apply(lambda x: (x=='False Negative').mean())
print(fn_by_age.round(3).sort_index())

print('\nFalse Negative rate by Race (among true <30):')
fn_by_race = X_test_err[X_test_err['y_true']==1].groupby('race')['error_type'].apply(lambda x: (x=='False Negative').mean())
print(fn_by_race.round(3))


## Q7 — Which features drive the model?

**Phase: Modeling & Evaluation**


In [ ]:
# Logistic Regression coefficients (most interpretable)
# Get feature names after one-hot
ohe = log_reg.named_steps['prep'].named_transformers_['cat']
cat_names = ohe.get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_names)

coefs = log_reg.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print('Top 15 features by absolute coefficient (Logistic Regression):')
print(coef_df.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
top = coef_df.head(15)
colors = ['#e74c3c' if c > 0 else '#3498db' for c in top['coefficient']]
ax.barh(top['feature'], top['coefficient'], color=colors)
ax.set_title('Top 15 Logistic Regression Coefficients\n(Red = increases early-readmit risk)')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '07_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Caution: Coefficients indicate association after controlling for other variables in the model. They are not causal effects.')


In [ ]:
# Permutation importance for Gradient Boosting
result = permutation_importance(gb, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='f1')
perm_df = pd.DataFrame({'feature': feature_cols, 'importance': result.importances_mean})
perm_df = perm_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=perm_df.head(12), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Permutation Importance (Gradient Boosting) — F1')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '07_gb_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(perm_df.head(12))


## Q8 — Is the model fair across patient subgroups?

**Phase: Modeling & Evaluation + Fairness**

We compare Recall and False Negative Rate across race, gender, and age brackets using the Logistic Regression model (primary interpretable model).


In [ ]:
def subgroup_performance(X_te, y_te, y_pr, group_col):
    results = []
    df_tmp = X_te.copy()
    df_tmp['y_true'] = y_te.values
    df_tmp['y_pred'] = y_pr
    for grp, sub in df_tmp.groupby(group_col):
        if len(sub) < 30:  # skip tiny groups
            continue
        rec = recall_score(sub['y_true'], sub['y_pred'], zero_division=0)
        fnr = 1 - rec  # False Negative Rate among positives
        prev = sub['y_true'].mean()
        n = len(sub)
        results.append({'Group': grp, 'N': n, 'Prevalence': round(prev,3),
                        'Recall': round(rec,3), 'FNR': round(fnr,3)})
    return pd.DataFrame(results).sort_values('Recall')

print('=== Fairness by Race ===')
fair_race = subgroup_performance(X_test, y_test, y_pred_lr, 'race')
print(fair_race.to_string(index=False))

print('\n=== Fairness by Gender ===')
fair_gender = subgroup_performance(X_test, y_test, y_pred_lr, 'gender')
print(fair_gender.to_string(index=False))

print('\n=== Fairness by Age ===')
fair_age = subgroup_performance(X_test, y_test, y_pred_lr, 'age')
print(fair_age.to_string(index=False))


In [ ]:
# Visualization of fairness
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=fair_race, x='Group', y='Recall', ax=axes[0], palette='Blues_d')
axes[0].set_title('Recall by Race')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylim(0, 1)

sns.barplot(data=fair_gender, x='Group', y='Recall', ax=axes[1], palette='Greens_d')
axes[1].set_title('Recall by Gender')
axes[1].set_ylim(0, 1)

sns.barplot(data=fair_age, x='Group', y='Recall', ax=axes[2], palette='Oranges_d')
axes[2].set_title('Recall by Age Bracket')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_PATH / '08_fairness_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Performance is not perfectly equal across subgroups. Differences should be reported transparently. Possible drivers include sample size imbalance, differing base rates, and residual confounding.')


## Q9 — What should a hospital administrator or clinician take from this analysis?

**Phase: Report Writing (گزارش‌نویسی)**


In [ ]:
print('''
==================== CLINICAL & OPERATIONAL SUMMARY ====================

Main patterns observed
• Early readmission (<30 days) occurs in ~11.2% of encounters.
• Strongest associated factors: higher number of prior inpatient visits, longer length of stay,
  higher medication count, and higher number of diagnoses.
• Demographic patterns exist but are weaker than utilization history.

Model performance (Logistic Regression – primary interpretable model)
• The model achieves modest discrimination. Overall accuracy is not the right metric.
• Recall for the critical <30 class is prioritized; the model surfaces a useful risk signal
  but is far from perfect.
• Gradient Boosting improves performance slightly, confirming non-linear structure in the data.

Fairness findings
• Recall and false-negative rates vary across race, gender, and age brackets.
• These differences are reported honestly. They may stem from sample size, base-rate differences,
  or unmeasured social determinants. The model should not be used to make unequal decisions
  across groups without further mitigation and prospective validation.

Limitations (critical)
• Data period: 1999–2008 (pre-many modern diabetes therapies and care pathways).
• ICD-9 coding only; modern systems use ICD-10.
• High missingness in weight, A1C, glucose serum, specialty, and payer.
• No socioeconomic, medication-adherence, or post-discharge support variables.
• Encounter-level (not purely patient-level) split; residual leakage risk remains.
• No external validation on contemporary data.

Actionable recommendations
1. Use the model only as a risk-stratification aid for enhanced discharge planning, not for
   denying care or automated decisions.
2. Prioritize patients with high prior inpatient utilization for medication reconciliation
   and early outpatient follow-up.
3. Before any operational deployment: retrain on recent data, add social determinants,
   perform prospective validation, and implement fairness monitoring.
4. Collect structured A1C and medication-adherence data going forward.

This is an educational risk-stratification exercise, not a deployed clinical decision system.
=======================================================================
''')


## Reproducibility & How to Run

```bash
pip install -r requirements.txt
cd notebooks
jupyter notebook analysis.ipynb
```

All figures are saved automatically to the `figures/` folder.

**Mapping of project phases to questions**

| Phase                        | Questions covered      |
|-----------------------------|------------------------|
| 1. آماده‌سازی داده‌ها       | Q1, Q2, Q3 + preprocessing |
| 2. مدل‌سازی و ارزیابی       | Q4, Q5, Q6, Q7, Q8     |
| 3. گزارش‌نویسی              | Q9 + README + summary  |
